# Lab Subsampling Analysis — Difference Maps

**Purpose:** Compute voxel-wise difference maps between max-dir reference and subsampled protocols  
**Data:** Lab subject, 3T multi-shell DWI (preprocessed with TOPUP+EDDY)  
**Protocols:** 14 total — b1000/b2000: n20,30,45,60,90 | b3000: n20,30,45,60  
**Maps computed:**
- DTI: FA, MD (reference = per-shell max-dir: b1000/b2000→n90, b3000→n60)
- NODDI: NDI, ODI, FWF (reference = per-shell max-dir: b1000/b2000→n90, b3000→n60)

**Difference definition:** `diff = reference - subsampled` (signed, masked to brain)  
**MAE:** Mean Absolute Error within brain mask  
**Subsampling SNR:** `mean(reference) / std(diff)` — analogous to tSNR; higher = closer to reference  

In [ ]:
#!/usr/bin/env python
"""
Lab subsampling difference map analysis
Author: Raiyun Kabir
Date: April 2026
"""

import os
import csv
import subprocess
import numpy as np
import nibabel as nib
from pathlib import Path
import time

print(f"NumPy:   {np.__version__}")
print(f"NiBabel: {nib.__version__}")

# Verify FSL is available
result = subprocess.run(['fslmaths'], capture_output=True)
print(f"FSL fslmaths: {'available' if result.returncode in [0,1] else 'NOT FOUND — check module load fsl'}")
result = subprocess.run(['fslstats'], capture_output=True)
print(f"FSL fslstats: {'available' if result.returncode in [0,1] else 'NOT FOUND — check module load fsl'}")

## Configuration

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

RESULTS_ROOT = "/scratch/rkabir5/StarterCodes_Data/results"
LAB_ROOT     = f"{RESULTS_ROOT}/Lab"

DTI_DIR   = f"{LAB_ROOT}/dtifit"
NODDI_DIR = f"{LAB_ROOT}/noddi"

MASK_FILE = "/scratch/rkabir5/StarterCodes_Data/new_data/afaiyaz-20260313_190724/TEST03092026/preproc/hifi_nodif_brain_mask.nii.gz"

OUTPUT_ROOT  = f"{LAB_ROOT}/analysis/diff_maps"
OUTPUT_DTI   = f"{OUTPUT_ROOT}/dtifit"
OUTPUT_NODDI = f"{OUTPUT_ROOT}/noddi"
TEMP_DIR     = f"{OUTPUT_ROOT}/tmp"

# b3000 max is 60dir (no 90dir for b3000 in Lab data)
PROTOCOLS_BY_SHELL = {
    1000: [20, 30, 45, 60, 90],
    2000: [20, 30, 45, 60, 90],
    3000: [20, 30, 45, 60],
}
DTI_MAX_DIRS   = {1000: 90, 2000: 90, 3000: 60}
NODDI_MAX_DIRS = {1000: 90, 2000: 90, 3000: 60}
NODDI_JOB_ID   = '962438'

PROTOCOLS = [
    {
        'shell':      shell,
        'ndirs':      ndirs,
        'label':      f'b{shell}_n{ndirs}',
        'dti_name':   f'b{shell}_{ndirs}dir',
        'noddi_name': f'noddi_b{shell}_n{ndirs}_{NODDI_JOB_ID}',
    }
    for shell, dirs_list in PROTOCOLS_BY_SHELL.items()
    for ndirs in dirs_list
]

Path(OUTPUT_DTI).mkdir(parents=True, exist_ok=True)
Path(OUTPUT_NODDI).mkdir(parents=True, exist_ok=True)
Path(TEMP_DIR).mkdir(parents=True, exist_ok=True)

print(f"Results root:  {RESULTS_ROOT}")
print(f"DTI input:     {DTI_DIR}")
print(f"NODDI input:   {NODDI_DIR}")
print(f"Output DTI:    {OUTPUT_DTI}")
print(f"Output NODDI:  {OUTPUT_NODDI}")
print(f"Temp dir:      {TEMP_DIR}")
print(f"Protocols:     {len(PROTOCOLS)} total")
for shell, dirs_list in PROTOCOLS_BY_SHELL.items():
    max_d = DTI_MAX_DIRS[shell]
    print(f"  b{shell}: dirs={dirs_list}  ref=b{shell}_{max_d}dir / noddi_b{shell}_n{max_d}_{NODDI_JOB_ID}")

## Verify Inputs

In [ ]:
print("Checking mask...")
assert os.path.exists(MASK_FILE), f"MISSING: {MASK_FILE}"
print(f"  OK: {MASK_FILE}")

print("\nChecking DTI per-shell references (max dirs)...")
for shell, max_d in DTI_MAX_DIRS.items():
    for metric in ['FA', 'MD']:
        f = f"{DTI_DIR}/b{shell}_{max_d}dir/dti_{metric}.nii.gz"
        assert os.path.exists(f), f"MISSING: {f}"
        print(f"  OK: b{shell}_{max_d}dir/dti_{metric}.nii.gz")

print("\nChecking NODDI per-shell references (max dirs)...")
for shell, max_d in NODDI_MAX_DIRS.items():
    for metric in ['NDI', 'ODI', 'FWF']:
        f = f"{NODDI_DIR}/noddi_b{shell}_n{max_d}_{NODDI_JOB_ID}/fit_{metric}.nii.gz"
        assert os.path.exists(f), f"MISSING: {f}"
        print(f"  OK: noddi_b{shell}_n{max_d}_{NODDI_JOB_ID}/fit_{metric}.nii.gz")

print("\nChecking all 14 subsampled DTI protocols...")
missing = []
for p in PROTOCOLS:
    for metric in ['FA', 'MD']:
        f = f"{DTI_DIR}/{p['dti_name']}/dti_{metric}.nii.gz"
        if not os.path.exists(f):
            missing.append(f)
if missing:
    print(f"  WARNING: {len(missing)} missing DTI files:")
    for m in missing: print(f"    {m}")
else:
    print(f"  All {len(PROTOCOLS)*2} DTI protocol files found.")

print("\nChecking all 14 subsampled NODDI protocols...")
missing = []
for p in PROTOCOLS:
    for metric in ['NDI', 'ODI', 'FWF']:
        f = f"{NODDI_DIR}/{p['noddi_name']}/fit_{metric}.nii.gz"
        if not os.path.exists(f):
            missing.append(f)
if missing:
    print(f"  WARNING: {len(missing)} missing NODDI files:")
    for m in missing: print(f"    {m}")
else:
    print(f"  All {len(PROTOCOLS)*3} NODDI protocol files found.")

print("\nInput verification complete.")

## Load Reference Maps

In [ ]:
# ---- Brain mask info (nibabel for display only) ----
print("Loading brain mask info...")
mask_img = nib.load(MASK_FILE)
mask_data = mask_img.get_fdata().astype(bool)
print(f"  Mask shape:   {mask_data.shape}")
print(f"  Brain voxels: {mask_data.sum():,}")
del mask_data  # not needed further — FSL uses the mask file directly


# ============================================================
# FSL helper functions
# ============================================================

def fsl_sub_mas(ref, sub, mask, out):
    """fslmaths ref -sub sub -mas mask -out out"""
    subprocess.run(
        ['fslmaths', ref, '-sub', sub, '-mas', mask, out],
        check=True, capture_output=True
    )

def fsl_mean(img, mask):
    """fslstats img -k mask -m  →  mean within mask"""
    r = subprocess.run(
        ['fslstats', img, '-k', mask, '-m'],
        check=True, capture_output=True, text=True
    )
    return float(r.stdout.strip())

def fsl_std(img, mask):
    """fslstats img -k mask -s  →  std within mask"""
    r = subprocess.run(
        ['fslstats', img, '-k', mask, '-s'],
        check=True, capture_output=True, text=True
    )
    return float(r.stdout.strip())

def fsl_mae(img, mask):
    """MAE = mean(|img|) within mask."""
    tmp = f"{TEMP_DIR}/_mae_tmp.nii.gz"
    subprocess.run(
        ['fslmaths', img, '-abs', tmp],
        check=True, capture_output=True
    )
    return fsl_mean(tmp, mask)


# ============================================================
# Precompute reference map means (for SNR numerator)
# 3 shells x 2 DTI metrics  = 6 fslstats calls
# 3 shells x 3 NODDI metrics = 9 fslstats calls
# ============================================================

print("\nPrecomputing DTI reference means (fslstats)...")
dti_ref_mean = {}
for shell, max_d in DTI_MAX_DIRS.items():
    dti_ref_mean[shell] = {}
    for metric in ['FA', 'MD']:
        ref_path = f"{DTI_DIR}/b{shell}_{max_d}dir/dti_{metric}.nii.gz"
        dti_ref_mean[shell][metric] = fsl_mean(ref_path, MASK_FILE)
        print(f"  b{shell}_{max_d}dir {metric} mean = {dti_ref_mean[shell][metric]:.5f}")

print("\nPrecomputing NODDI reference means (fslstats)...")
noddi_ref_mean = {}
for shell, max_d in NODDI_MAX_DIRS.items():
    noddi_ref_mean[shell] = {}
    for metric in ['NDI', 'ODI', 'FWF']:
        ref_path = f"{NODDI_DIR}/noddi_b{shell}_n{max_d}_{NODDI_JOB_ID}/fit_{metric}.nii.gz"
        noddi_ref_mean[shell][metric] = fsl_mean(ref_path, MASK_FILE)
        print(f"  b{shell}_n{max_d} {metric} mean = {noddi_ref_mean[shell][metric]:.5f}")

print("\nReference means precomputed.")

## DTI Difference Maps

`diff = ref_maxdir_same_shell - subsampled`  
Positive: subsampled underestimates. Negative: subsampled overestimates.  
**SNR** = `mean(ref[mask]) / std(diff[mask])` — higher is better.

In [ ]:
print("Computing DTI difference maps (fslmaths + fslstats)...")
print("=" * 70)

dti_mae = {}
dti_snr = {}
t0 = time.time()

for p in PROTOCOLS:
    label = p['label']
    shell = p['shell']
    max_d = DTI_MAX_DIRS[shell]
    dti_mae[label] = {}
    dti_snr[label] = {}

    for metric in ['FA', 'MD']:
        ref_path = f"{DTI_DIR}/b{shell}_{max_d}dir/dti_{metric}.nii.gz"
        sub_path = f"{DTI_DIR}/{p['dti_name']}/dti_{metric}.nii.gz"
        out_path = f"{OUTPUT_DTI}/diff_{metric}_{label}.nii.gz"

        fsl_sub_mas(ref_path, sub_path, MASK_FILE, out_path)
        dti_mae[label][metric] = fsl_mae(out_path, MASK_FILE)

        std_diff = fsl_std(out_path, MASK_FILE)
        dti_snr[label][metric] = (
            dti_ref_mean[shell][metric] / std_diff
            if std_diff > 0 else np.inf
        )

    print(
        f"  {label:15s}  "
        f"FA  MAE={dti_mae[label]['FA']:.5f} SNR={dti_snr[label]['FA']:6.1f}  "
        f"MD  MAE={dti_mae[label]['MD']:.2e} SNR={dti_snr[label]['MD']:6.1f}"
    )

print("=" * 70)
print(f"Done. Elapsed: {time.time()-t0:.1f}s")
print(f"Saved {len(PROTOCOLS)*2} DTI difference maps to: {OUTPUT_DTI}")

## NODDI Difference Maps

`diff = noddi_b{shell}_n{maxdir} - subsampled` (per-shell max-dir reference)  
Positive: subsampled underestimates. Negative: subsampled overestimates.  
**SNR** = `mean(ref[mask]) / std(diff[mask])`

In [ ]:
print("Computing NODDI difference maps (fslmaths + fslstats)...")
print("=" * 80)

noddi_mae = {}
noddi_snr = {}
t0 = time.time()

for p in PROTOCOLS:
    label = p['label']
    shell = p['shell']
    max_d = NODDI_MAX_DIRS[shell]
    noddi_mae[label] = {}
    noddi_snr[label] = {}

    for metric in ['NDI', 'ODI', 'FWF']:
        ref_path = f"{NODDI_DIR}/noddi_b{shell}_n{max_d}_{NODDI_JOB_ID}/fit_{metric}.nii.gz"
        sub_path = f"{NODDI_DIR}/{p['noddi_name']}/fit_{metric}.nii.gz"
        out_path = f"{OUTPUT_NODDI}/diff_{metric}_{label}.nii.gz"

        fsl_sub_mas(ref_path, sub_path, MASK_FILE, out_path)
        noddi_mae[label][metric] = fsl_mae(out_path, MASK_FILE)

        std_diff = fsl_std(out_path, MASK_FILE)
        noddi_snr[label][metric] = (
            noddi_ref_mean[shell][metric] / std_diff
            if std_diff > 0 else np.inf
        )

    print(
        f"  {label:15s}  "
        f"NDI MAE={noddi_mae[label]['NDI']:.5f} SNR={noddi_snr[label]['NDI']:6.1f}  "
        f"ODI MAE={noddi_mae[label]['ODI']:.5f} SNR={noddi_snr[label]['ODI']:6.1f}  "
        f"FWF MAE={noddi_mae[label]['FWF']:.5f} SNR={noddi_snr[label]['FWF']:6.1f}"
    )

print("=" * 80)
print(f"Done. Elapsed: {time.time()-t0:.1f}s")
print(f"Saved {len(PROTOCOLS)*3} NODDI difference maps to: {OUTPUT_NODDI}")

## Summary — Tables and CSV

In [ ]:
# ---- Build rows ----
rows = []
for p in PROTOCOLS:
    label = p['label']
    rows.append({
        'protocol': label,
        'shell':    p['shell'],
        'ndirs':    p['ndirs'],
        'FA_MAE':   dti_mae[label]['FA'],
        'FA_SNR':   dti_snr[label]['FA'],
        'MD_MAE':   dti_mae[label]['MD'],
        'MD_SNR':   dti_snr[label]['MD'],
        'NDI_MAE':  noddi_mae[label]['NDI'],
        'NDI_SNR':  noddi_snr[label]['NDI'],
        'ODI_MAE':  noddi_mae[label]['ODI'],
        'ODI_SNR':  noddi_snr[label]['ODI'],
        'FWF_MAE':  noddi_mae[label]['FWF'],
        'FWF_SNR':  noddi_snr[label]['FWF'],
    })

# ---- Save CSV ----
csv_file = f"{OUTPUT_ROOT}/lab_mae_snr_summary.csv"
fieldnames = [
    'protocol', 'shell', 'ndirs',
    'FA_MAE', 'FA_SNR', 'MD_MAE', 'MD_SNR',
    'NDI_MAE', 'NDI_SNR', 'ODI_MAE', 'ODI_SNR', 'FWF_MAE', 'FWF_SNR',
]
with open(csv_file, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)
print(f"CSV saved: {csv_file}")

# ---- MAE table ----
print()
W = 83
print("=" * W)
print("LAB SUMMARY -- Mean Absolute Error (MAE) within brain mask")
print("=" * W)
print(f"{'Protocol':<15} {'FA_MAE':>8} {'MD_MAE(e-4)':>12} {'NDI_MAE':>9} {'ODI_MAE':>9} {'FWF_MAE':>9}")
print("-" * W)
for shell in PROTOCOLS_BY_SHELL:
    for r in [x for x in rows if x['shell'] == shell]:
        print(
            f"{r['protocol']:<15} "
            f"{r['FA_MAE']:>8.5f} "
            f"{r['MD_MAE']*1e4:>12.4f} "
            f"{r['NDI_MAE']:>9.5f} "
            f"{r['ODI_MAE']:>9.5f} "
            f"{r['FWF_MAE']:>9.5f}"
        )
    print("-" * W)

# ---- SNR table ----
print()
print("=" * W)
print("LAB SUMMARY -- Subsampling SNR  [SNR = mean(ref) / std(diff)]")
print("  Higher SNR = subsampled protocol closer to reference")
print("  Ref: Jones DK (2004), MRM 51(4):807-815")
print("=" * W)
print(f"{'Protocol':<15} {'FA_SNR':>8} {'MD_SNR':>8} {'NDI_SNR':>9} {'ODI_SNR':>9} {'FWF_SNR':>9}")
print("-" * W)
for shell in PROTOCOLS_BY_SHELL:
    for r in [x for x in rows if x['shell'] == shell]:
        print(
            f"{r['protocol']:<15} "
            f"{r['FA_SNR']:>8.1f} "
            f"{r['MD_SNR']:>8.1f} "
            f"{r['NDI_SNR']:>9.1f} "
            f"{r['ODI_SNR']:>9.1f} "
            f"{r['FWF_SNR']:>9.1f}"
        )
    print("-" * W)

# ---- File counts ----
print()
dti_count   = len(list(Path(OUTPUT_DTI).glob('*.nii.gz')))
noddi_count = len(list(Path(OUTPUT_NODDI).glob('*.nii.gz')))
print(f"Output location:       {OUTPUT_ROOT}")
print(f"DTI diff maps saved:   {dti_count}")
print(f"NODDI diff maps saved: {noddi_count}")
print(f"Total NIfTI files:     {dti_count + noddi_count}")
print(f"CSV summary:           lab_mae_snr_summary.csv")

# ---- Cleanup temp dir ----
import shutil
shutil.rmtree(TEMP_DIR)
print(f"\nTemp directory cleaned up: {TEMP_DIR}")